<center><h1>Yu_Stella_HW3</h1></center>
<br>
<br>

Name: Stella Yu
<br>
Github Username: syu35435-cmyk
<br>
USC ID: 3420585291

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [1]:
import os
import re
import numpy as np
import pandas as pd
from sklearn.utils import resample

Get the AReM Data Set

In [2]:
columns = ["time", "avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23"]

df = pd.read_csv("data/AReM/AReM/bending1/dataset1.csv",
                 skiprows=4, usecols=range(7))

df.columns = columns
df.head()

,time,avg_rss12,var_rss12,avg_rss13,var_rss13,avg_rss23,var_rss23
0,0,39.25,0.43,22.75,0.43,33.75,1.3
1,250,39.25,0.43,23.00,0.00,33.00,0.0
2,500,39.25,0.43,23.25,0.43,33.00,0.0
3,750,39.50,0.50,23.00,0.71,33.00,0.0
4,1000,39.50,0.50,24.00,0.00,33.00,0.0


### (b) Test and Train Data

In [3]:
activities = ["bending1", "bending2", "cycling", "lying","sitting", "standing", "walking"]

train_data = []
test_data = []

for activity in activities:
    folder = "data/AReM/AReM/" + activity

    for file in os.listdir(folder):

        if activity == "bending1" or activity == "bending2":
            if file == "dataset1.csv" or file == "dataset2.csv":
                test_data.append(folder + "/" + file)
            elif file.endswith(".csv"):
                train_data.append(folder + "/" + file)
    
        else:
            if file == "dataset1.csv" or file == "dataset2.csv" or file == "dataset3.csv":
                test_data.append(folder + "/" + file)
            elif file.endswith(".csv"):
                train_data.append(folder + "/" + file)

### (c) Feature Extraction

#### i. Research

They include minimum, maximum, mean, median, standard deviation, variance, first quartile, third quartile, range, and interquartile range.

#### ii. Extraction

In [4]:
rows = []

files = train_data + test_data

time_series = ["avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23"]

for file in files:
    df = pd.read_csv(file, skiprows=4, usecols=range(7))

    row = []

    # extract features
    for column in time_series:
        row.append(df[column].min())
        row.append(df[column].max())
        row.append(df[column].mean())
        row.append(df[column].median())
        row.append(df[column].std())
        row.append(df[column].quantile(0.25))
        row.append(df[column].quantile(0.75))

    rows.append(row)

columns = []

for i in range(1, 7):
    columns.append("min" + str(i))
    columns.append("max" + str(i))
    columns.append("mean" + str(i))
    columns.append("median" + str(i))
    columns.append("std" + str(i))
    columns.append("1st_quart" + str(i))
    columns.append("3rd_quart" + str(i))

feature_df = pd.DataFrame(rows, columns=columns)

feature_df

,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,mean2,...,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
0,35.00,47.40,43.954500,44.330,1.558835,43.00,45.0000,0.0,1.70,0.426250,...,1.999604,35.3625,36.500,0.0,1.79,0.493292,0.430,0.513506,0.000,0.940
1,33.00,47.75,42.179812,43.500,3.670666,39.15,45.0000,0.0,3.00,0.696042,...,3.849448,30.4575,36.330,0.0,2.18,0.613521,0.500,0.524317,0.000,1.000
2,33.00,45.75,41.678063,41.750,2.243490,41.33,42.7500,0.0,2.83,0.535979,...,2.411026,28.4575,31.250,0.0,1.79,0.383292,0.430,0.389164,0.000,0.500
3,37.00,48.00,43.454958,43.250,1.386098,42.50,45.0000,0.0,1.58,0.378083,...,2.488862,22.2500,24.000,0.0,5.26,0.679646,0.500,0.622534,0.430,0.870
4,36.25,48.00,43.969125,44.500,1.618364,43.31,44.6700,0.0,1.50,0.413125,...,3.318301,20.5000,23.750,0.0,2.96,0.555312,0.490,0.487826,0.000,0.830
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,35.50,46.25,43.174938,43.670,1.989052,42.50,44.5000,0.0,2.12,0.506583,...,2.983976,12.7500,16.500,0.0,5.72,0.911979,0.830,0.666161,0.470,1.220
84,32.75,47.00,42.760562,44.500,3.398919,41.33,45.3725,0.0,3.34,0.486167,...,4.296574,13.0000,18.565,0.0,5.73,0.842271,0.710,0.722165,0.430,1.090
85,19.33,43.50,34.227771,35.500,4.889576,30.50,37.7500,0.0,14.50,3.995729,...,3.092094,14.7500,18.670,0.0,9.74,3.394125,3.100,1.792090,2.105,4.425
86,12.50,45.00,33.509729,34.125,4.850923,30.50,36.7500,0.0,13.05,4.450771,...,3.133564,14.6275,18.750,0.0,8.96,3.378479,3.085,1.787360,2.060,4.440


#### iii. Standard Deviation

In [5]:
results = []

for column in feature_df.columns:
    data = feature_df[column]
    std = data.std()

    bootstrap_std = []

    for i in range(1000):
        sample = resample(data)
        bootstrap_std.append(sample.std())

    bootstrap_std.sort()

    # 90% confidence interval
    lower = bootstrap_std[49]
    upper = bootstrap_std[949]

    results.append([std, lower, upper])

std_df = pd.DataFrame(
    results,
    index=feature_df.columns,
    columns=["Standard Deviation", "Lower Bound", "Upper Bound"]
)

std_df

,Standard Deviation,Lower Bound,Upper Bound
min1,9.624011,8.168972,10.902471
max1,4.207745,3.110476,5.117862
mean1,5.276431,4.638766,5.827545
median1,5.386624,4.692147,5.964755
std1,1.771282,1.566026,1.953161
1st_quart1,6.127846,5.534657,6.643897
3rd_quart1,5.031028,4.225819,5.781625
min2,0.000000,0.000000,0.000000
max2,5.059656,4.589728,5.403148
mean2,1.577908,1.403281,1.713589


#### iv. Select Features

I choose min, mean, and max as the three most important features becaus these features gives both the average and the extremities of the time series. 

## 2. ISLR 3.7.4

### (a) Linear Train

Cubic regression is expected to have a lower training RSS because it can fit to the training data better due to its flexibiliy.

### (b) Linear Test

Linear regression is expected to have a lower test RSS because the cubit regression may overfit to the linear relationship due to its flexibility and higher variance.

### (c) Not Linear Train

Cubic regression is expected to have a lower training RSS because its flexibility makes it better at fitting to the data compared to linear regression. 

### (d) Not Linear Testing

There is not enough information to tell because there is no information on how the relationship is. Cubic regression may fit the nonlinear relationship better, but it can also result in higher variance. 

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 